In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

# 1. 패키지로드 & 한글설정 & 경고메세지 ignore

In [ ]:
# 패키지 import
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 시각화의 선명도를 높임
%config InlineBackend.figure_format = "retina"
# from IPython.display import set_matplotlib_formats
# set_matplotlib_formats('retina')

from matplotlib.font_manager import FontProperties, fontManager
font_path = r"C:\Users\Admin\Downloads\JMK체.ttf"
fontManager.addfont(font_path)
font_prop = FontProperties(fname=font_path, size=12)
font_name = font_prop.get_name()

# 한글설정
sns.set(
    style="white",
    context="notebook",
    palette="Dark2",
    rc={
        "figure.figsize": (12, 3),
        "font.family": font_name,  # 한글 폰트 지정
        "axes.unicode_minus": False,  # 마이너스 부호 깨짐 방지
    },
)

plt.rcParams['axes.unicode_minus'] = False

# warning(경고) 안보이게
import warnings
# warnings.filterwarnings(action='ignore') # 경고 메세지 안보이게
# warnings.filterwarnings(action='default') # 경고 메세지 보이게

# 2. 서울과 부산를 df변수에 불러오기

In [ ]:
# 큰 용량의 데이터 불러오기 시 발생하는 타입 에러 해결방법 1
df_seoul = pd.read_csv(r"C:\ai\downloads\shareData\상가정보\소상공인시장진흥공단_상가(상권)정보_서울_202603.csv", 
                       low_memory=False # 파일을 chunk 단위로 나누지 않고, 전체 내용을 한번에 불러옴 => 타입 일관성 유지
                      )
df_seoul.shape

In [ ]:
# 큰 용량의 데이터 불러오기 시 발생하는 타입 에러 해결방법 2 : 방법 1에 비하여 메모리 관리 효율이 높음
df_seoul = pd.read_csv(r"C:\ai\downloads\shareData\상가정보\소상공인시장진흥공단_상가(상권)정보_서울_202603.csv", 
                       dtype={'층정보':str} # 에러가 발생한 위치에 대해 타입을 지정 => 타입 일관성 유지
                      )
df_seoul.shape

In [ ]:
df_seoul['층정보'].unique()[10:20] # 일반적인 read_csv로 chunk 단위로 불러올 시, float과 str이 혼재됨

In [ ]:
# 상세 메모리 사용량
df_seoul.info(memory_usage='deep')

In [ ]:
df_busan = pd.read_csv(r"C:\ai\downloads\shareData\상가정보\소상공인시장진흥공단_상가(상권)정보_부산_202603.csv", 
                       dtype={'층정보':str})
df_busan.shape

In [ ]:
df_busan.head(1).T

In [ ]:
# 서울시 소상공인 데이터와 부산시 소상공인 데이터 concat을 위한 준비

In [ ]:
# 컬럼명 일치 여부 확인
np.all(df_seoul.columns == df_busan.columns)

In [ ]:
# 컬럼 타입 일치 여부 확인 - 건물본번지 불일치 확인 / 서울 - float64, 부산 - int64
np.all(df_seoul.dtypes == df_busan.dtypes)

In [ ]:
df_seoul['건물본번지'].dtype

In [ ]:
df_busan['건물본번지'].astype('float64')

In [ ]:
# 서울시 건물본번지가 결측치인 데이터
df_seoul.loc[df_seoul['건물본번지'].isna(),['상호명','건물본번지']]

In [ ]:
# 서울시 소상공인 데이터와 부산시 소상공인 데이터 concat
df = pd.concat([df_seoul, df_busan]).reset_index(drop=True)

In [ ]:
del df_seoul, df_busan

# 3. df 데이터 셋의 결측치 및 시각화
## ① df 변수의 컬럼 확인 후 상위 3줄, 하위3줄을 출력

In [ ]:
df.head(3)

In [ ]:
df.tail(3)

## ②	df 변수의 결측치를 제외한 데이터 개수 및 dtype들을 출력 후 변수가 사용하는 메모리 사용량 확인

In [ ]:
df.info(memory_usage='deep')

## ③	결측치를 확인하고 결측치가 없는 컬럼을 포함하여 시각화(정렬 전 bar plot, barh plot, 정렬 후 bar plot, barh plot)

In [ ]:
missing_cnt = df.isna().sum()
missing_cnt.head()

In [ ]:
missing_cnt_sorted = missing_cnt.sort_values(ascending=False)

In [ ]:
missing_cnt_sorted.plot(kind='bar', figsize=(20,5))

In [ ]:
missing_cnt.sort_values().plot(kind='barh', figsize=(10,10))

## ④ 결측치가 없는 컬럼을 제외하고 시각화(정렬전 bar plot, barh plot, 정렬 후 bar plot, barh plot).

In [ ]:
only_missing_cnt = missing_cnt[missing_cnt!=0]    

In [ ]:
ax = only_missing_cnt.sort_values().plot(kind='bar', rot=0, figsize=(15,5), ylim=[0,750000])
plt.ylabel

labels = ax.bar_label(ax.containers[0], fmt="%.0f", padding=3, fontsize=12)

for bar, label in zip(ax.containers[0], labels):
    if bar.get_height() < 100000:
        label.set_color("red")
    else:
        label.set_color("blue")

plt.show()

In [ ]:
result = only_missing_cnt.to_frame()
result.columns = ['결측치 개수']
result.head()

In [ ]:
sns.barplot(x=only_missing_cnt.index, y=only_missing_cnt)

for i, v in enumerate(only_missing_cnt):
    if v>100000:
        plt.text(i, v, round(v), va='bottom', ha='center', size = 12, color='b')
    else:
        plt.text(i, v, round(v), va='bottom', ha='center', size = 12, color='r')

plt.xticks(rotation=45)        
plt.ylim(0,750000)
plt.grid('y')
plt.show()

## ⑤ missingno 라이브러리로 결측치 시각화
* [ResidentMario/missingno: Missing data visualization module for Python](https://github.com/ResidentMario/missingno)

* 위의 사이트를 열어 사용법 확인
`pip install missingno`

In [ ]:
import missingno as msno

# 데이터 : 검은색, 결측치 : 흰색 표현
msno.matrix(df)

In [ ]:
# 결측치가 아닌 데이터 개수
msno.bar(df)

# 4. 사용하지 않을 컬럼 제거
## ①결측치가 너무 많은 컬럼은 제거. (결측치 개수 상위 7개 컬럼 제거)

In [ ]:
# 결측치 개수 상위 7개 컬럼 
not_use = df.isnull().sum().sort_values(ascending=False).head(7)

In [ ]:
# 결측치 비율
round(not_use / df.shape[0] * 100, 2)

In [ ]:
# 컬럼 삭제 전 - 메모리 용량 : 206.1+ MB / 컬럼 수 : 39
print('컬럼 제거 전 shape :', df.shape)
# 컬럼 삭제
df = df.drop(not_use.index, axis=1)
# 컬럼 삭제 후 - 메모리 용량 : 169.1+ MB / 컬럼 수 : 32
print('컬럼 제거 후 shape :', df.shape)

## ②	컬럼명에 “코드”나 “번호”가 있는 컬럼 제거

In [ ]:
cols = df.columns
drop_cols = df[cols[cols.str.contains('코드|번호') == 1]]

**※ 시리즈에 문자함수를 쓰기 위해 참조 :**
https://pandas.pydata.org/pandas-docs/stable/reference/series.html#string-handling

In [ ]:
df.drop(drop_cols, axis=1, inplace=True)
# 컬럼 삭제 후 - 메모리 용량 : 89.8+ MB / 컬럼 수 : 17
print('컬럼 제거 후 shape :', df.shape)

# ※ df 파일i/o

In [ ]:
df.to_csv(r'C:\ai\downloads\shareData\상가정보\서울부산상가정보백업.csv', index=False)

In [ ]:
df.to_csv(r'C:\ai\downloads\shareData\상가정보\서울부산상가정보백업.zip', index=False,  compression='infer')

In [ ]:
df = pd.read_csv(r"C:\ai\downloads\shareData\상가정보\서울부산상가정보백업.csv", low_memory=False)

In [ ]:
df.shape

# 5. df  값 호출하기(loc함수, iloc함수 등)
## ① 상호명 필드 호출


In [ ]:
df['상호명']

In [ ]:
df.loc[:,'상호명']

In [ ]:
df.iloc[:,0]

## ② 상호명 필드의 종류별 데이터 수(df.상호명.value_counts()이용)

In [ ]:
df['상호명'].unique()

In [ ]:
df.groupby('상호명')['경도'].count().sort_values(ascending=False).head()

In [ ]:
df['상호명'].value_counts()

## ③	“상호명”과 "도로명주소” 컬럼 호출

In [ ]:
df[['상호명', '도로명주소']]

## ④ 0~2행 출력 (loc과 iloc을 이용)

In [ ]:
df.loc[:2]

In [ ]:
df.iloc[:3]

# 6. 기술 통계값 보기 

- [Descriptive statistics - Wikipedia](https://en.wikipedia.org/wiki/Descriptive_statistics)
- [Computations / descriptive stats](https://pandas.pydata.org/docs/reference/frame.html#computations-descriptive-stats)
- [표준 편차 - 위키백과, 우리 모두의 백과사전](https://ko.wikipedia.org/wiki/표준편차)
* describe() 사용 시 데이터 요약을 볼 수 있음 ; 기술통계량
    - 기본적으로 수치형 데이터를 요약
    - include, exclude 옵션으로 다른 데이터 타입의 요약 수치 출력
    - count, min, max, mean, median, 1사분위수, 3사분위수

### **개별 기술 통계값 구하기**

* [Computations / descriptive stats](https://pandas.pydata.org/docs/reference/frame.html#computations-descriptive-stats)
* count : 결측치를 제외한 값 갯수
* min, max: 최솟값, 최댓값
* argmin, argmax : 최솟값 인덱스, 최댓값 인덱스 반환
* quantile : 특정 사분위수에 해당하는 값을 반환 (0~1 사이)
    * 0.25 : 1사분위 수
    * 0.5 : 2사분위수 (quantile 기본 값)
    * 0.75 : 3사분위수
* sum : 수치 데이터의 합계
* mean : 평균
* median : 중앙값(중간값:데이터를 한 줄로 세웠을 때 가운데 위치하는 값, 중앙값이 짝수일 때는 가운데 2개 값의 평균을 구함)

* std, var : 표준편차, 분산을 계산
* cumsum : 맨 첫 번째 성분부터 각 성분까지의 누적합을 계산 (0 번째 부터 계속 더해짐)
* cumprod : 맨 첫번째 성분부터 각 성분까지의 누적곱을 계산 (1 번째 부터 계속 곱해짐)
* cov, corr() : 공분산, 상관계수
* skew : 왜도 (오른쪽으로 치우침= 왜도<0, 왼쪽으로 치우침=왜도>0)
* kurt : 첨도


## ①	df의 요약기술 통계량

In [ ]:
# 수치형 데이터에 대한 기술통계량
df.describe()

In [ ]:
# object형 데이터에 대한 기술 통계량
df.describe(include='object')

## ②	“지번본번지", "건물본번지” 컬럼의 데이터 개수와  dtype, 메모리 사용량을 확인

In [ ]:
df[['지번본번지','건물본번지']].count()

In [ ]:
df[['지번본번지','건물본번지']].info(memory_usage='deep')

## ③	“위도", "경도” 컬럼의 요약 기술통계량

In [ ]:
df[['위도', '경도']].describe()

# 7.단별량 수치형 변수 시각화
## ① 위도의 빈도표를 시각화 : plot.hist, hist, sns.displot, sns.histplot

In [ ]:
sns.displot(data=df, x='위도')

In [ ]:
df['위도'].plot(kind='hist', bins=100)

In [ ]:
plt.hist(df['위도'], bins=100)
plt.show()

In [ ]:
sns.histplot(data=df, x='위도')
plt.axvline(df['위도'].mean(), color='r')
plt.annotate('평균', xytext=(36.7, 10000), 
             xy=(df['위도'].mean(),5000),
             arrowprops = dict(arrowstyle='->',
             connectionstyle='angle3',
             color='k'),
             fontsize=15)
plt.axvline(df['위도'].median(), color='b')
plt.annotate('중위수', xytext=(37.2, 10000), 
             xy=(df['위도'].median(),5000),
             arrowprops = dict(arrowstyle='->',
             connectionstyle='angle',
             color='k'),
             fontsize=15)

## ② 경도의 빈도표를 시각화 : plot.hist, hist, sns.displot, sns.histplot

In [ ]:
sns.histplot(data=df, x='경도')
plt.axvline(df['경도'].mean(), color='r')
plt.annotate('평균', xytext=(127.6, 10000), 
             xy=(df['경도'].mean(),5000),
             arrowprops = dict(arrowstyle='-|>',
             connectionstyle='angle3',
             color='k'),
             fontsize=15)
plt.axvline(df['경도'].median(), color='b')
plt.annotate('중위수', xytext=(127.2, 10000), 
             xy=(df['경도'].median(),5000),
             arrowprops = dict(arrowstyle='-|>',
             connectionstyle='angle',
             color='k'),
             fontsize=15)
plt.show()

In [ ]:
plt.hist(df['경도'], bins=100)
plt.axvline(df['경도'].mean(), color='r')
plt.annotate('평균', xytext=(127.6, 50000), 
             xy=(df['경도'].mean(),20000),
             arrowprops = dict(arrowstyle='-|>',
             connectionstyle='angle3',
             color='k'),
             fontsize=15)
plt.axvline(df['경도'].median(), color='b')
plt.annotate('중위수', xytext=(127.2, 50000), 
             xy=(df['경도'].median(),20000),
             arrowprops = dict(arrowstyle='-|>',
             connectionstyle='angle',
             color='k'),
             fontsize=15)
plt.show()

In [ ]:
df['위도'].skew()

In [ ]:
df['위도'].kurt()

# 8. 상관계수
-  두 변수 간에 어떤 선형적 관계를 갖고 있는 지를 분석하는 방법
- [상관 분석 - 위키백과, 우리 모두의 백과사전M](https://ko.wikipedia.org/wiki/상관_분석) 

- 결과의 해석 : r 값은 X 와 Y 가 완전히 동일할 때 +1, 전혀 다르면 0, 반대방향으로 완전히 동일하면 –1 을 가진다. 결정계수(coefficient of determination)는 $r^2$로 계산하며 이것은 X 로부터 Y를 예측할 수 있는 정도를 의미한다.
- ([수학기호 사이트 참조](https://matplotlib.org/2.0.2/users/mathtext.html))

- 일반적으로
    * r이 -1.0과 -0.7 사이이면, 강한 음적 선형관계,
    * r이 -0.7과 -0.3 사이이면, 뚜렷한 음적 선형관계,
    * r이 -0.3과 -0.1 사이이면, 약한 음적 선형관계,
    * r이 -0.1과 +0.1 사이이면, 무시할 수 있는 선형관계,
    * r이 +0.1과 +0.3 사이이면, 약한 양적 선형관계,
    * r이 +0.3과 +0.7 사이이면, 뚜렷한 양적 선형관계,
    * r이 +0.7과 +1.0 사이이면, 강한 양적 선형관계로 해석한다.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/d/d4/Correlation_examples2.svg/1280px-Correlation_examples2.svg.png" width="600">

이미지 출처 : [위키백과](https://ko.wikipedia.org/wiki/상관_분석)


## ① 전체 숫자 컬럼끼리의 상관계수

In [ ]:
corr = df.corr(numeric_only=True)

## ②	상관계수를 이용하여 heatmap 시각화
- 참조 http://seaborn.pydata.org/examples/many_pairwise_correlations.html 

In [ ]:
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='twilight')

In [ ]:
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=~mask, annot=True, fmt='.2f', cmap='twilight')

# 9.	경도와 위도 컬럼을 이용하여 산점도

[Anscombe’s quartet — seaborn 0.10.0 documentation](https://seaborn.pydata.org/examples/anscombes_quartet.html)

## ①	X축에는 경도, y축에는 위도 컬럼을 산점도로 시각화

In [ ]:
plt.figure(figsize=(4,6))
ax=sns.scatterplot(data=df.sample(frac=0.1), x='경도', y='위도', hue='시도명', s=3)
ax.set_title('산점도', color='royalblue', fontsize=15)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.show()

## ②	경도와 위도의 산점도를 “시도명” 컬럼별로 서브플롯으로 시각화

In [ ]:
sns.relplot(kind='scatter', data=df.sample(frac=0.8), marker='^', x='경도', y='위도', hue='시도명', s=10, col='시도명',
            facet_kws={'sharex':False,'sharey':False}, height=6, aspect=1, legend=False, col_order=['서울특별시', '부산광역시'], alpha=0.5, palette=['skyblue','violet'])

plt.show()

## ③ 위의 1번에서 시각화된 산점도를 바탕으로 회귀선을 그린다.

In [ ]:
sns.regplot(data=df.sample(frac=0.5), x='경도', y='위도')

plt.show()

## ④ 위의 2번에서 시각화된 산점도를 바탕으로 회귀선을 서브 플롯으로 그린다.

In [ ]:
sns.lmplot(data=df.sample(frac=0.1), x='경도', y='위도', hue='시도명', col='시도명', palette='terrain',
            facet_kws={'sharex':False,'sharey':False}, height=6, aspect=1, legend=False, scatter_kws={"alpha": 0.3, "s": 20}, line_kws={"color": "red", "linewidth": 2})

plt.show()

# 10. 상권업종대분류명별 상호명의 개수를 도출하고 시각화하기

In [ ]:
all(df['상권업종대분류명'].isna())

In [ ]:
df_1 = df.groupby('상권업종대분류명')['상호명'].count().sort_values(ascending=False).to_frame()
df_1

In [ ]:
result = df.pivot_table(index='상권업종대분류명',
              values='상호명',
              aggfunc='count').sort_values(by='상호명', ascending=False).reset_index()
result

In [ ]:
sns.countplot(data=df, x='상권업종대분류명', order=df_1.index)

In [ ]:
df.groupby('상권업종대분류명')['상호명'].count().sort_values(ascending=False).plot(kind='bar', rot=0)

In [ ]:
result.plot(kind='bar', rot = 0)
plt.xticks(
    ticks=range(len(result)),
    labels=result['상권업종대분류명'])
plt.show()

In [ ]:
ax = sns.barplot(data=result,
           x='상권업종대분류명',
           y='상호명')
labels = ax.bar_label(ax.containers[0], fmt="%.0f", padding=3, fontsize=12)
plt.ylim([0,220000])
plt.show()

In [ ]:
group = df.groupby(['상권업종대분류명', '상권업종중분류명', '상권업종소분류명'])
print(len(group))

In [ ]:
for (major, medium, sub), df_val in group:
    if major == '음식':
        print(f'{major} - {medium} - {sub}')

# 11. 상권업종대분류명이 음식인 서브셋 분석
## ① “상권업종대분류명”이 음식인 서브셋을 변수 df_food에 할당 후 확인


In [ ]:
df_food = df[df['상권업종대분류명']=='음식']
df_food.shape

In [ ]:
df.info(memory_usage='deep')

## ②	“상권업종대분류명”이 음식이면서, "시군구명”이 강남구인 데이터만 가져와 “상권업종중분류명”별 빈도수 확인(loc함수, etc)

In [ ]:
# 구별 상호명수
df_food.groupby('시군구명')['상호명'].count().sort_values(ascending=False)
df_food['시군구명'].value_counts() # 결측치가 없을 때 사용 가능

In [ ]:
df_food.loc[df_food['시군구명']=='강남구', '상권업종중분류명'].value_counts()

In [ ]:
df_food[df_food['시군구명']=='강남구']['상권업종중분류명'].value_counts()

# 12. “상권업종대분류명”이 음식인 데이터 중 서울특별시 데이터 서브셋 생성
- https://seaborn.pydata.org/tutorial/categorical.html : 범주형그래프

## ①“상권업종대분류명”이 음식인 서브셋 중 서울특별시 데이터만 변수 df_seoul_food에 할당 후 확인

In [ ]:
# 시리즈의 and 연산은 '&'만 가능함
df_seoul_food = df[(df['시도명']=='서울특별시') & (df['상권업종대분류명']=='음식')]
df_seoul_food.shape

In [ ]:
df_seoul_food

## ② df_seoul_food를 시군구명, 상권업종중분류명으로 그룹화하여 상점수 count한 내용을 food_gu 변수에 할당

In [ ]:
food_gu = df_seoul_food.groupby(['시군구명','상권업종중분류명'])['상호명'].count()

## ③ food_gu 변수를 표로 출력(unstack)

In [ ]:
food_gu.unstack().replace(np.nan, '')

## ④ 위 3번 스타일의 표를 pivot_table함수를 이용하여 출력

In [ ]:
df_seoul_food.pivot_table(index='시군구명', columns='상권업종중분류명', values='상호명', aggfunc='count').replace(np.nan,'')

## ⑤ 3번의 결과 중 강남구 데이터만 뽑아 barplot으로 시각화(판다스 plot이용)

In [ ]:
ax=food_gu['강남구'].sort_values(ascending=False).plot(kind='bar', rot = 0)
labels = ax.bar_label(ax.containers[0], fmt="%.0f", padding=3, fontsize=12)
plt.ylim([0,5500])
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)

## ⑥ food_gu를 seaborn을 이용하여 구별 음식점 상호 개수 시각화

In [ ]:
food_gu_df = food_gu.reset_index().rename(columns={'상호명':'상호수'})
food_gu_df

In [ ]:
g=sns.catplot(data=food_gu_df, x='시군구명', y='상호수', kind='bar', errorbar=None,
             order = food_gu_df.groupby('시군구명')['상호수'].sum().sort_values().index)
plt.xticks(rotation=45)
g.fig.set_size_inches(12,3)
plt.show()

In [ ]:
g=sns.catplot(data=food_gu_df.sort_values('상호수', ascending=False), x='시군구명', y='상호수', kind='bar', errorbar=None, hue='시군구명', col='상권업종중분류명', col_wrap=1, 
              dodge=False, sharey=False, sharex=False, height=2, aspect=8)

## ⑦ 상권업종중분류명별 음식점 상호수

In [ ]:
food_gu_df.groupby('상권업종중분류명')['상호수'].sum()
df_seoul_food['상권업종중분류명'].value_counts()

## ⑧	Seaborn의 catplot을 이용하여 상권업종중분류별 음식점을 구별로 시각화(서브플롯으로 시각화)

https://seaborn.pydata.org/tutorial/categorical.html

In [ ]:
sns.catplot(data=food_gu_df, kind='bar', x='상권업종중분류명', y='상호수', estimator='sum', errorbar=None, height=3, aspect=4, order=df_seoul_food['상권업종중분류명'].value_counts().index)

## ⑨	Seaborn의 catplot을 이용하여 구별 음식점을 상권업종중분류명별로 서브 플롯으로 시각화

In [ ]:
sns.barplot(data=food_gu_df, x='상권업종중분류명', y='상호수', hue='시군구명')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.show()

In [ ]:
g=sns.catplot(data=food_gu_df.sort_values('상호수', ascending=False), kind='bar', x='상권업종중분류명', y='상호수', hue='상권업종중분류명', col='시군구명', col_wrap=2, 
              dodge=False, sharey=False, sharex=False, height=2, aspect=4, estimator='sum')
g.set_xticklabels(rotation=45, fontweight='bold')
g.tight_layout()
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., ncol=2)
plt.show()